# 📰 Hacker News Historical Data Collector

Fetches the full HN archive using the **Algolia HN Search API** and stores results as a CSV on GitHub.

## Architecture
```
Algolia HN Search API  →  this notebook  →  GitHub (CSV)
       (free, no key)                         (public access)
```

## Modes
| Section | What it does | Needs token? |
|---------|-------------|-------------|
| 4 | Full historical fetch (2010 → today) | Yes |
| 5 | READ — load stored data from GitHub | No (public repo) |
| 7 | Incremental update (new stories only) | Yes |

## API reference
- Base URL: `https://hn.algolia.com/api/v1/`
- Rate limit: 10,000 requests/hour — free, no key needed
- Algolia hard-caps results at **page 999** per query → we work around this by chunking by year

---
## 0. Install dependencies

In [1]:
# Uncomment and run once if any package is missing
# !pip install requests pandas tqdm python-dotenv

---
## 1. Configuration

In [2]:
import os
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()  # reads .env file in the same folder as this notebook

# ── GitHub settings ────────────────────────────────────────────────────────
GITHUB_REPO  = "annhmartin/dataviz-historical-stocks-AnnetteMartin"  # username/repo
GITHUB_PATH  = "hn_stories.csv"       # path inside the repo
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", None)

# ── Query settings ─────────────────────────────────────────────────────────
QUERY        = ""          # empty = all stories; or e.g. "machine learning"
TAGS         = "story"     # story | comment | ask_hn | show_hn | poll
MIN_POINTS   = 0           # set > 0 to filter low-signal stories
HISTORY_FROM = 2010        # start year for full historical fetch

print("Configuration loaded ✓")
print(f"  Repo   : {GITHUB_REPO}")
print(f"  Path   : {GITHUB_PATH}")
print(f"  Token  : {'set ✓' if GITHUB_TOKEN else 'NOT SET — read-only mode'}")

Configuration loaded ✓
  Repo   : annhmartin/dataviz-historical-stocks-AnnetteMartin
  Path   : hn_stories.csv
  Token  : set ✓


---
## 2. Algolia HN API helpers

In [3]:
import requests
import time
import pandas as pd

HN_BASE = "https://hn.algolia.com/api/v1"
HITS_PER_PAGE = 1000   # Algolia maximum


def _date_to_unix(date_str: str) -> int:
    dt = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    return int(dt.timestamp())


def _hits_to_df(hits: list) -> pd.DataFrame:
    rows = []
    for h in hits:
        rows.append({
            "id"           : h.get("objectID"),
            "title"        : h.get("title"),
            "url"          : h.get("url"),
            "author"       : h.get("author"),
            "points"       : h.get("points"),
            "num_comments" : h.get("num_comments"),
            "created_at"   : h.get("created_at"),
            "created_at_i" : h.get("created_at_i"),
            "story_text"   : h.get("story_text", ""),
            "tags"         : ", ".join(h.get("_tags", [])),
            "hn_url"       : f"https://news.ycombinator.com/item?id={h.get('objectID')}",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
    return df


def fetch_year(
    year: int,
    query: str = "",
    tags: str = "story",
    min_points: int = 0,
    delay: float = 0.25,
) -> pd.DataFrame:
    """
    Fetch ALL stories for a single calendar year.

    Algolia caps results at page 999 (1,000 hits/page × 1,000 pages = 1M max).
    A single year of HN stories is well within that limit, so chunking by year
    guarantees we retrieve everything without hitting the cap.
    """
    date_from = f"{year}-01-01"
    date_to   = f"{year}-12-31"

    numeric_filters = [
        f"created_at_i>={_date_to_unix(date_from)}",
        f"created_at_i<={_date_to_unix(date_to) + 86399}",
    ]
    if min_points > 0:
        numeric_filters.append(f"points>={min_points}")

    params = {
        "query"          : query,
        "tags"           : tags,
        "hitsPerPage"    : HITS_PER_PAGE,
        "numericFilters" : ",".join(numeric_filters),
        "page"           : 0,
    }

    all_hits = []
    page = 0

    while True:
        params["page"] = page
        resp = requests.get(
            f"{HN_BASE}/search_by_date", params=params, timeout=30
        )
        resp.raise_for_status()
        data      = resp.json()
        hits      = data.get("hits", [])
        nb_pages  = data.get("nbPages", 1)
        nb_hits   = data.get("nbHits", 0)

        if page == 0:
            print(f"    {year}: {nb_hits:,} stories across {nb_pages} pages", flush=True)

        all_hits.extend(hits)
        print(f"      page {page + 1}/{nb_pages} — {len(all_hits):,} collected so far", end="\r", flush=True)

        if page >= nb_pages - 1 or not hits:
            break

        page += 1
        time.sleep(delay)

    print(f"      ✓ {year} complete — {len(all_hits):,} stories        ")
    return _hits_to_df(all_hits)


print("HN API helpers loaded ✓")

HN API helpers loaded ✓


---
## 3. GitHub storage helpers

In [13]:
import base64
import io

GITHUB_API = "https://api.github.com"


def _gh_headers(token: str) -> dict:
    return {
        "Authorization"       : f"Bearer {token}",
        "Accept"              : "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }


def push_csv_to_github(
    df: pd.DataFrame,
    repo: str,
    path: str,
    token: str,
    commit_message: str | None = None,
) -> str:
    """Push a DataFrame as CSV to GitHub. Creates or updates the file."""
    if commit_message is None:
        ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
        commit_message = f"Update HN data [{ts}] — {len(df):,} rows"

    buf = io.StringIO()
    df.to_csv(buf, index=False)
    encoded = base64.b64encode(buf.getvalue().encode()).decode()

    url     = f"{GITHUB_API}/repos/{repo}/contents/{path}"
    headers = _gh_headers(token)

    existing_sha = None
    check = requests.get(url, headers=headers, timeout=15)
    if check.status_code == 200:
        existing_sha = check.json().get("sha")

    payload = {"message": commit_message, "content": encoded}
    if existing_sha:
        payload["sha"] = existing_sha

    resp = requests.put(url, headers=headers, json=payload, timeout=60)
    resp.raise_for_status()

    raw_url = f"https://raw.githubusercontent.com/{repo}/main/{path}"
    print(f"  ✓ Pushed {len(df):,} rows → {raw_url}")
    return raw_url


def load_csv_from_github(repo: str, path: str, token: str | None = None) -> pd.DataFrame:
    """Load a CSV from GitHub using the raw URL — works for large files."""
    raw_url = f"https://raw.githubusercontent.com/{repo}/main/{path}"
    print(f"  Loading from: {raw_url}")
    
    headers = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    
    resp = requests.get(raw_url, headers=headers, timeout=60)
    
    if resp.status_code == 404:
        raise FileNotFoundError(
            f"No file at {repo}/{path}. Run Section 4 first to create it."
        )
    resp.raise_for_status()
    
    content = resp.text.strip()
    if not content:
        print("File is empty — treating as no data.")
        return pd.DataFrame()
    
    df = pd.read_csv(io.StringIO(content), parse_dates=["created_at"])
    print(f"  ✓ Loaded {len(df):,} rows from {repo}/{path}")
    return df


print("GitHub helpers loaded ✓")

GitHub helpers loaded ✓


---
## 4. FULL HISTORICAL FETCH — 2010 → today

> Run this **once** to build the complete dataset.
> It fetches one year at a time and pushes to GitHub after each year so progress is never lost.
> Expect it to take **20–60 minutes** depending on your internet speed and the volume of stories.
> If it gets interrupted, re-run — it will skip years already stored and resume from where it left off.

In [5]:
if GITHUB_TOKEN is None:
    print("GITHUB_TOKEN not set — cannot fetch. Set it in Section 1.")
else:
    current_year = datetime.now(timezone.utc).year
    years        = list(range(HISTORY_FROM, current_year + 1))

    # ── Load whatever is already stored on GitHub ──────────────────────────
    try:
        df_existing = load_csv_from_github(GITHUB_REPO, GITHUB_PATH, GITHUB_TOKEN)
        df_existing["created_at"] = pd.to_datetime(df_existing["created_at"], utc=True)
        already_done = set(df_existing["created_at"].dt.year.unique())
        print(f"  Years already stored: {sorted(already_done)}")
    except FileNotFoundError:
        df_existing  = pd.DataFrame()
        already_done = set()
        print("  No existing data — starting fresh.")

    # ── Fetch missing years one at a time ──────────────────────────────────
    years_to_fetch = [y for y in years if y not in already_done]
    print(f"\n🔍 Years to fetch: {years_to_fetch}\n")

    all_frames = [df_existing] if not df_existing.empty else []

    for year in years_to_fetch:
        df_year = fetch_year(
            year       = year,
            query      = QUERY,
            tags       = TAGS,
            min_points = MIN_POINTS,
        )

        if df_year.empty:
            print(f"  {year}: no results — skipping.")
            continue

        all_frames.append(df_year)

        # Combine everything fetched so far and push after each year
        df_so_far = (
            pd.concat(all_frames, ignore_index=True)
            .drop_duplicates(subset="id")
            .sort_values("created_at")
            .reset_index(drop=True)
        )

        push_csv_to_github(
            df             = df_so_far,
            repo           = GITHUB_REPO,
            path           = GITHUB_PATH,
            token          = GITHUB_TOKEN,
            commit_message = f"Add {year} data ({len(df_year):,} stories) — total {len(df_so_far):,} rows",
        )

    print("\n🎉 Full historical fetch complete!")
    print(f"   Total rows in dataset: {len(df_so_far):,}")
    df = df_so_far   # make available for Section 6

  No existing data — starting fresh.

🔍 Years to fetch: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

    2010: 31,206 stories across 1 pages
      ✓ 2010 complete — 1,000 stories        
  ✓ Pushed 1,000 rows → https://raw.githubusercontent.com/annhmartin/dataviz-historical-stocks-AnnetteMartin/main/hn_stories.csv
    2011: 32,111 stories across 1 pages
      ✓ 2011 complete — 1,000 stories        
  ✓ Pushed 2,000 rows → https://raw.githubusercontent.com/annhmartin/dataviz-historical-stocks-AnnetteMartin/main/hn_stories.csv
    2012: 33,225 stories across 1 pages
      ✓ 2012 complete — 1,000 stories        
  ✓ Pushed 3,000 rows → https://raw.githubusercontent.com/annhmartin/dataviz-historical-stocks-AnnetteMartin/main/hn_stories.csv
    2013: 34,792 stories across 1 pages
      ✓ 2013 complete — 1,000 stories        
  ✓ Pushed 4,000 rows → https://raw.githubusercontent.com/annhmartin/dataviz-historical-stocks-AnnetteMartin/

---
## 5. READ mode — load stored data from GitHub

Anyone can run this cell — no credentials needed for a public repo.

In [8]:
print("📥 Loading stored dataset from GitHub …")

raw_url = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_PATH}"
print(f"Loading from: {raw_url}")
df = pd.read_csv(raw_url, parse_dates=["created_at"])
print(f"✓ Loaded {len(df):,} rows")
df.head()

📥 Loading stored dataset from GitHub …
Loading from: https://raw.githubusercontent.com/annhmartin/dataviz-historical-stocks-AnnetteMartin/main/hn_stories.csv
✓ Loaded 17,000 rows


,id,title,url,author,points,num_comments,created_at,created_at_i,story_text,tags,hn_url
0,2049477,How to Design Programs: An Introduction to Com...,http://htdp.org/2003-09-26/Book/,zoowar,121,17,2010-12-29 17:40:23+00:00,1293644423,NaN,"story, author_zoowar, story_2049477",https://news.ycombinator.com/item?id=2049477
1,2049479,Forged in the Stars - a love letter to NASA,http://www.loe.org/shows/segments.htm?programI...,limist,1,0,2010-12-29 17:40:29+00:00,1293644429,NaN,"story, author_limist, story_2049479",https://news.ycombinator.com/item?id=2049479
2,2049481,"North American English Dialects, Based on Pron...",http://aschmann.net/AmEng/#Southern,eggspurt,2,1,2010-12-29 17:41:20+00:00,1293644480,NaN,"story, author_eggspurt, story_2049481",https://news.ycombinator.com/item?id=2049481
3,2049484,Pinboard.In Architecture - Pay To Play To Keep...,http://highscalability.com/blog/2010/12/29/pin...,yarapavan,6,0,2010-12-29 17:42:06+00:00,1293644526,NaN,"story, author_yarapavan, story_2049484",https://news.ycombinator.com/item?id=2049484
4,2049496,Why your child's school bus has no seat belts,http://www.msnbc.msn.com/id/40820669/ns/us_new...,thedoctor,171,104,2010-12-29 17:45:24+00:00,1293644724,NaN,"story, author_thedoctor, story_2049496",https://news.ycombinator.com/item?id=2049496


---
## 6. Quick exploration

In [9]:
print("Dataset Overview")
print(f"Rows       : {len(df):,}")
print(f"Columns    : {list(df.columns)}")
print(f"Date range : {df['created_at'].min()}  →  {df['created_at'].max()}")
print()
print(df[["points", "num_comments"]].describe().round(1))

=== Dataset overview ===
Rows       : 17,000
Columns    : ['id', 'title', 'url', 'author', 'points', 'num_comments', 'created_at', 'created_at_i', 'story_text', 'tags', 'hn_url']
Date range : 2010-12-29 17:40:23+00:00  →  2026-06-27 19:01:09+00:00

        points  num_comments
count  17000.0       17000.0
mean      17.9           9.0
std       63.7          39.9
min        0.0           0.0
25%        1.0           0.0
50%        2.0           0.0
75%        5.0           1.0
max     1721.0        1188.0


In [10]:
# Top 10 stories by points
(
    df[["title", "author", "points", "num_comments", "created_at", "hn_url"]]
    .sort_values("points", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

,title,author,points,num_comments,created_at,hn_url
0,"IRS Reforms Free File Program, Drops Agreement...",danso,1721,448,2019-12-31 18:31:44+00:00,https://news.ycombinator.com/item?id=21923220
1,In Memoriam: Ian Murdock,spb,1457,366,2015-12-30 18:43:55+00:00,https://news.ycombinator.com/item?id=10813524
2,Happy New Year HN!,thunderbong,1393,265,2023-12-31 18:33:34+00:00,https://news.ycombinator.com/item?id=38826283
3,What I Didn't Say,twampss,1289,567,2013-12-30 20:40:49+00:00,https://news.ycombinator.com/item?id=6986797
4,Happy New Year 2025,martynvandijke,1223,258,2024-12-31 23:48:36+00:00,https://news.ycombinator.com/item?id=42562750
5,Draw SVG rope using JavaScript,stanko,1211,66,2022-12-31 15:43:54+00:00,https://news.ycombinator.com/item?id=34197379
6,U.S. government will decide who gets to use GP...,alain94040,1132,1188,2026-06-26 18:23:14+00:00,https://news.ycombinator.com/item?id=48690101
7,"Tell HN: John Friel my father, internet pionee...",AaronFriel,1092,98,2024-12-30 18:11:47+00:00,https://news.ycombinator.com/item?id=42551900
8,Previewing GPT‑5.6 Sol: a next-generation model,minimaxir,1090,695,2026-06-26 17:06:55+00:00,https://news.ycombinator.com/item?id=48689028
9,Happy New Year HN,thunderbong,986,236,2020-12-31 19:01:27+00:00,https://news.ycombinator.com/item?id=25595865


In [11]:
# Stories per year
df["year"] = pd.to_datetime(df["created_at"], utc=True).dt.year
df.groupby("year").agg(
    stories=("id", "count"),
    avg_pts=("points", "mean"),
).round(1)

,stories,avg_pts
year,,
2010,1000,12.5
2011,1000,12.4
2012,1000,12.4
2013,1000,14.1
2014,1000,14.4
2015,1000,19.6
2016,1000,16.9
2017,1000,20.0
2018,1000,20.8


---
## 7. Incremental update — append new stories only

Run this after the initial full fetch to keep the dataset current.
It detects the latest stored date and only fetches stories newer than that.

In [14]:
if GITHUB_TOKEN is None:
    print("GITHUB_TOKEN not set — skipping.")
else:
    # ── Find where the stored data ends ───────────────────────────────────
    df_stored = load_csv_from_github(GITHUB_REPO, GITHUB_PATH, GITHUB_TOKEN)
    df_stored["created_at"] = pd.to_datetime(df_stored["created_at"], utc=True)

    latest_dt = df_stored["created_at"].max()
    new_from  = latest_dt.strftime("%Y-%m-%d")
    new_to    = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    print(f"Fetching new stories: {new_from} → {new_to}")

    # ── Fetch only the new window ──────────────────────────────────────────
    # If the window spans multiple years, fetch year by year
    start_year = int(new_from[:4])
    end_year   = int(new_to[:4])
    new_frames = []

    for year in range(start_year, end_year + 1):
        df_year = fetch_year(
            year       = year,
            query      = QUERY,
            tags       = TAGS,
            min_points = MIN_POINTS,
        )
        if not df_year.empty:
            new_frames.append(df_year)

    if not new_frames:
        print("No new stories found.")
    else:
        df_new = pd.concat(new_frames, ignore_index=True)

        # ── Merge, deduplicate, sort ───────────────────────────────────────
        df_combined = (
            pd.concat([df_stored, df_new], ignore_index=True)
            .drop_duplicates(subset="id")
            .sort_values("created_at")
            .reset_index(drop=True)
        )
        added = len(df_combined) - len(df_stored)
        print(f"  Added {added:,} new rows — total now {len(df_combined):,}")

        push_csv_to_github(
            df             = df_combined,
            repo           = GITHUB_REPO,
            path           = GITHUB_PATH,
            token          = GITHUB_TOKEN,
            commit_message = f"Incremental update {new_from} → {new_to} (+{added:,} rows)",
        )
        df = df_combined

  Loading from: https://raw.githubusercontent.com/annhmartin/dataviz-historical-stocks-AnnetteMartin/main/hn_stories.csv
  ✓ Loaded 17,000 rows from annhmartin/dataviz-historical-stocks-AnnetteMartin/hn_stories.csv
Fetching new stories: 2026-06-27 → 2026-06-27
    2026: 3,724,815 stories across 1 pages
      ✓ 2026 complete — 1,000 stories        
  Added 1,000 new rows — total now 18,000
  ✓ Pushed 18,000 rows → https://raw.githubusercontent.com/annhmartin/dataviz-historical-stocks-AnnetteMartin/main/hn_stories.csv


---
## 8. Quick reference

| Parameter | Values | Notes |
|-----------|--------|-------|
| `tags` | `story`, `comment`, `ask_hn`, `show_hn`, `poll` | Can combine: `ask_hn,story` |
| `query` | any string | Empty = all items |
| `numericFilters` | `created_at_i>=X`, `points>=N` | UNIX timestamps |
| `hitsPerPage` | 1–1000 | Use 1000 for bulk fetches |
| `page` | 0-indexed, max 999 | → chunk by year to avoid the cap |

**Endpoints**:
- `search_by_date` — chronological order (best for historical work)
- `search` — sorted by relevance

In [15]:
print("finished")

finished
